In [0]:
API_KEY = dbutils.secrets.get(scope="tc_02", key="api_key")
API_SECRET = dbutils.secrets.get(scope="tc_02", key="api_secret")
BOOTSTRAP_SERVER = dbutils.secrets.get(scope="tc_02", key="bootstrap_servers")

bootstrap = BOOTSTRAP_SERVER if ":" in BOOTSTRAP_SERVER else f"{BOOTSTRAP_SERVER}:9092"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS tc02.testKafka;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS tc02.testKafka.checkpoints;
CREATE VOLUME IF NOT EXISTS tc02.testKafka.origin;

In [0]:
display(spark.sql("SHOW SCHEMAS IN tc02"))

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{API_KEY}" password="{API_SECRET}";'
    ),
    "subscribe": "alunos-eventos",
    "startingOffsets": "earliest",
}

df = spark.readStream.format("kafka").options(**kafka_options).load()
df_parsed = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "timestamp")

checkpoint_path = "/Volumes/tc02/testKafka/checkpoints/alunos_eventos_v3"
origin_path = "/Volumes/tc02/testKafka/origin/alunos_eventos"

query = (
    df_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(origin_path)
)

query.processAllAvailable()
print("Lote processado. Status final:", query.status)

In [0]:
display(spark.read.format("delta").load(origin_path))

In [0]:
print(type(BOOTSTRAP_SERVER), repr(BOOTSTRAP_SERVER))

In [0]:
display(spark.sql("SHOW CATALOGS"))